# 11  Re-label the OCR'd comments 

In [9]:
import os, json, time
import pandas as pd
from openai import OpenAI

GEMINI_KEY = ""

GEMINI_MODEL = "gemini-2.5-flash-lite"
OCR_MAIN_CSV = "cleaned_comments_ver2_with_ocr.csv"   # from nb09 cell 3
UNIQ_CSV     = "full_llm_UNIQUE.csv"                   # existing LLM labels
OUT_OCR_CSV  = "full_llm_OCR.csv"                      
OUT_ALL_CSV  = "full_llm_ALL_with_ocr.csv"            # final per-comment labels for the app

TOPIC_CODES = ["oil_spill_risk","climate_emissions","air_quality","environmental_justice",
               "marine_wildlife","fisheries","wetlands_coast","national_interest"]
TOPIC_DESC = {
 "oil_spill_risk":"oil spills, leaks, blowouts, spill response, Deepwater Horizon, catastrophic accident",
 "climate_emissions":"climate change, greenhouse gas, CO2, fossil fuel expansion, emissions",
 "air_quality":"air pollution, ozone, NOx, local air health",
 "environmental_justice":"impact on low-income / minority / coastal communities, fairness, EJ",
 "marine_wildlife":"sea turtles, marine mammals, dolphins, whales, birds, habitat, ecosystems",
 "fisheries":"fishing industry, shrimp, commercial fishing, fishermen, fishery impacts",
 "wetlands_coast":"wetlands, coastline, erosion, storm surge, sea-level rise, flooding",
 "national_interest":"whether the project is / isn't in the national interest (general opinion)",
}
topic_block = "\n".join(f"- {c}: {TOPIC_DESC[c]}" for c in TOPIC_CODES)
SYSTEM = f"""You label public comments on a US offshore oil port (GulfLink).
Assign ALL applicable TOPIC codes (multi-label) from this fixed list ONLY:
{topic_block}

Also give STANCE = the commenter's position on the project:
- oppose  = against the project (incl. raising harms/pollution as reasons to reject)
- support = in favour
- unclear = neutral / mixed / question only / no position

Return ONLY raw JSON, no prose, no code fences:
{{"topics": ["code", ...], "stance": "oppose|support|unclear", "confidence": 0.0, "uncertain": false}}
Rules: topics must be from the list; if none apply -> []. If you cannot decide topic -> uncertain:true, topics:[]."""

client = OpenAI(api_key=GEMINI_KEY,
                base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
                timeout=60, max_retries=0)
VALID = set(TOPIC_CODES)

def llm_label(text):
    text = str(text)[:8000]           
    last_err = None
    for k in range(3):
        try:
            r = client.chat.completions.create(model=GEMINI_MODEL, temperature=0,
                messages=[{"role": "system", "content": SYSTEM},
                          {"role": "user", "content": text}])
            raw = (r.choices[0].message.content or "").strip().replace("```json", "").replace("```", "").strip()
            o = json.loads(raw)
            o["topics"] = [t for t in o.get("topics", []) if t in VALID]
            o["FAILED"] = False; o["err"] = ""
            return o
        except Exception as e:
            last_err = str(e); time.sleep(min(2 ** k, 8))
    return {"topics": [], "stance": "unclear", "confidence": 0.0,
            "uncertain": True, "FAILED": True, "err": last_err or "unknown"}

print("setup OK — key set:", GEMINI_KEY != "")

setup OK — key set: False


In [11]:
from tqdm.auto import tqdm

df = pd.read_csv(OCR_MAIN_CSV, low_memory=False)
todo = df[df["ocr_text"].fillna("").astype(str).str.len() > 0].copy()
print(f"comments with OCR text to re-label: {len(todo)}")

prev = pd.read_csv(OUT_OCR_CSV) if os.path.exists(OUT_OCR_CSV) else None
done = set(prev["Document ID"].astype(str)) if prev is not None else set()
todo = todo[~todo["Document ID"].astype(str).isin(done)]
print(f"remaining after resume: {len(todo)}")

recs = prev.to_dict("records") if prev is not None else []
new, failed = 0, 0
for _, row in tqdm(todo.iterrows(), total=len(todo), desc="re-label", unit="doc"):
    o = llm_label(row["llm_input_text"])
    recs.append({"Document ID": row["Document ID"],
                 "llm_topics": "|".join(o["topics"]) or "NONE",
                 "llm_stance": o.get("stance", "unclear"),
                 "llm_conf": o.get("confidence", 0.0),
                 "llm_failed": o.get("FAILED", False),
                 "labeled_from": "ocr"})
    new += 1; failed += int(o.get("FAILED", False))
    if new % 10 == 0:
        pd.DataFrame(recs).to_csv(OUT_OCR_CSV, index=False)
    time.sleep(0.2)
pd.DataFrame(recs).to_csv(OUT_OCR_CSV, index=False)
print(f"done -> {OUT_OCR_CSV}: {len(recs)} comments, {failed} failed")

comments with OCR text to re-label: 178
remaining after resume: 178


re-label:   0%|          | 0/178 [00:00<?, ?doc/s]

done -> full_llm_OCR.csv: 178 comments, 0 failed


In [12]:
df   = pd.read_csv(OCR_MAIN_CSV, low_memory=False)
uniq = pd.read_csv(UNIQ_CSV)
ocrl = pd.read_csv(OUT_OCR_CSV)

base = df[["Document ID", "comment_text"]].merge(
        uniq[["comment_text", "llm_topics", "llm_stance", "llm_conf"]], on="comment_text", how="left")
base["labeled_from"] = "text"

o = ocrl.drop_duplicates("Document ID", keep="last").set_index("Document ID")
base = base.set_index("Document ID")
idx = o.index.intersection(base.index)
for c in ["llm_topics", "llm_stance", "llm_conf"]:
    base.loc[idx, c] = o.loc[idx, c]
base.loc[idx, "labeled_from"] = "ocr"
base = base.reset_index()

base[["Document ID", "llm_topics", "llm_stance", "llm_conf", "labeled_from"]].to_csv(OUT_ALL_CSV, index=False)
print(f"wrote {OUT_ALL_CSV}: {len(base)} rows | overridden by OCR: {int((base['labeled_from'] == 'ocr').sum())}")
print(base["labeled_from"].value_counts().to_dict())

wrote full_llm_ALL_with_ocr.csv: 10195 rows | overridden by OCR: 178
{'text': 10017, 'ocr': 178}
